# 02 · CNN training — Pneumonia classification (EfficientNet-B0 + Grad-CAM)
**Owner:** M3 (ML Engineer — Diagnosis) · **Drop 3**

**Before running on Kaggle**
1. *Settings → Accelerator → GPU T4 x2 (or P100)*, *Internet → On* (needed once for ImageNet weights).
2. *Add data →* `paultimothymooney/chest-xray-pneumonia` (5,863 images, NORMAL / PNEUMONIA).
3. Attach the repo code (see first cell).

**Outputs:** `models/best_cnn.pth`, `results/training_curves.png`, `results/confusion_matrix.png`,
`results/roc_curve.png`, `results/gradcam_samples.png`, `results/cnn_test_metrics.json`

**Target:** test accuracy ≥ 85 %

In [ ]:
# --- get the repo code into the Kaggle session ---------------------------------
# Option 1 (repo is private): upload the repo as a Kaggle *Dataset* named "medical-ai-code"
#   and attach it -> it appears under /kaggle/input/medical-ai-code
# Option 2: !git clone https://<TOKEN>@github.com/<team>/medical-ai.git
import os, sys, shutil, glob
CODE_SRC = next(iter(glob.glob("/kaggle/input/**/src/diagnosis", recursive=True)), None)
if CODE_SRC and not os.path.exists("/kaggle/working/medical-ai"):
    shutil.copytree(os.path.dirname(os.path.dirname(CODE_SRC)), "/kaggle/working/medical-ai")
REPO = "/kaggle/working/medical-ai" if os.path.exists("/kaggle/working/medical-ai") else os.path.abspath("..")
os.chdir(REPO); sys.path.insert(0, REPO)
print("repo:", REPO)

In [ ]:
!pip -q install albumentations scikit-learn opencv-python-headless 2>/dev/null | tail -1

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))   # screenshot this for the group (Drop 1)

## 1 · Data — patient-level 70/15/15 split
The official `val/` folder has only 16 images, so all images are pooled and re-split **by patient** (no patient in two splits). If M2 has produced `cxr_split.csv`, set `CSV` instead.

In [ ]:
import glob, os
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/train/NORMAL", recursive=True)]
DATA_ROOT = sorted(cands, key=len)[0] if cands else "data/raw/chest_xray"   # first match
CSV = None   # e.g. "data/splits/cxr_split.csv" from M2
print("DATA_ROOT =", DATA_ROOT)

from src.data.dataset import load_cxr_table
df = load_cxr_table(DATA_ROOT, CSV)
display(df.groupby(["split", "label"]).size().unstack())
print("patients per split:", df.groupby("split").patient_id.nunique().to_dict())

In [ ]:
# Drop-2 check: 2 batches load correctly
import matplotlib.pyplot as plt
from src.data.dataset import make_cls_loaders
from src.preprocessing.augment import denormalize
loaders, cw, _ = make_cls_loaders(DATA_ROOT, CSV, batch_size=8, num_workers=2)
x, y = next(iter(loaders["train"]))
print("batch:", tuple(x.shape), "labels:", y.tolist(), "class weights:", cw.tolist())
fig, ax = plt.subplots(1, 8, figsize=(18, 2.6))
for i in range(8):
    ax[i].imshow(denormalize(x[i])); ax[i].set_title(["NORMAL", "PNEUMONIA"][y[i]], fontsize=8); ax[i].axis("off")
plt.show()

## 2 · Train (5 epochs, AdamW, class-weighted CrossEntropyLoss, AMP) — ~25–35 min on T4

In [ ]:
!python -m src.diagnosis.train_classifier --data_root "$DATA_ROOT" --epochs 5 --bs 32 --lr 3e-4 --out models/best_cnn.pth

In [ ]:
from IPython.display import Image, display
display(Image("results/training_curves.png"))

## 3 · Evaluate on the held-out TEST patients
`--tune_threshold` picks the decision threshold on **validation** (Youden J) — the test set is never used for tuning.

In [ ]:
!python -m src.diagnosis.evaluate --ckpt models/best_cnn.pth --data_root "$DATA_ROOT" --tune_threshold --out_dir results

In [ ]:
import json
res = json.load(open("results/cnn_test_metrics.json"))
import pandas as pd
display(pd.DataFrame(res).drop("confusion_matrix").T)
display(Image("results/confusion_matrix.png")); display(Image("results/roc_curve.png"))
print(open("results/classification_report.txt").read())

## 4 · Grad-CAM (2 samples: one NORMAL, one PNEUMONIA)

In [ ]:
!python -m src.diagnosis.gradcam --ckpt models/best_cnn.pth --data_root "$DATA_ROOT" --out results/gradcam_samples.png
display(Image("results/gradcam_samples.png"))

## 5 · Inference function used by the API (Drop 7)

In [ ]:
from src.diagnosis.inference import predict_classification
sample = df[(df.split == "test") & (df.label == "PNEUMONIA")].path.iloc[0]
r = predict_classification(sample)
print({k: v for k, v in r.items() if k != "gradcam_png_base64"})

## 6 · Save artefacts
Download `/kaggle/working/m3_cnn_outputs.zip` → push `results/*` to GitHub, give `best_cnn.pth` to M2 for the Drive backup (weights are **not** committed to git).

In [ ]:
!cd /kaggle/working && zip -qr m3_cnn_outputs.zip medical-ai/models/best_cnn.pth medical-ai/models/cnn_history.json medical-ai/results && ls -lh m3_cnn_outputs.zip

## 7 · Results (fill in after the run)
| Metric (test) | Value |
|---|---|
| Accuracy | |
| Precision | |
| Recall / Sensitivity | |
| Specificity | |
| F1 | |
| AUROC | |
| Threshold (from val) | |